In [1]:
import pandas as pd
import os

In [2]:
def read_file(file_name:str):
    df = (pd.read_csv(f"data/after-2016/{file_name}.csv", sep=";", encoding='latin-1')).drop(["id"],axis=1)
    return df

In [3]:
# Nota: confundi after com antes, logo esses dados na verdade refletem dados de 2016 para trás (2016-2007)
file_names = os.listdir("data/after-2016")

In [4]:
file_names[0]

'datatran2007.csv'

In [5]:
dfs = []
for i in file_names:
    file_name = i.split(".")[0]
    dfs.append(read_file(file_name))

C:\Users\RAFAEL\AppData\Local\Temp\ipykernel_21084\3648755116.py:2: DtypeWarning: Columns (0: br, 1: km) have mixed types. Specify dtype option on import or set low_memory=False.
  df = (pd.read_csv(f"data/after-2016/{file_name}.csv", sep=";", encoding='latin-1')).drop(["id"],axis=1)
C:\Users\RAFAEL\AppData\Local\Temp\ipykernel_21084\3648755116.py:2: DtypeWarning: Columns (0: br, 1: km) have mixed types. Specify dtype option on import or set low_memory=False.
  df = (pd.read_csv(f"data/after-2016/{file_name}.csv", sep=";", encoding='latin-1')).drop(["id"],axis=1)
C:\Users\RAFAEL\AppData\Local\Temp\ipykernel_21084\3648755116.py:2: DtypeWarning: Columns (0: br, 1: km) have mixed types. Specify dtype option on import or set low_memory=False.
  df = (pd.read_csv(f"data/after-2016/{file_name}.csv", sep=";", encoding='latin-1')).drop(["id"],axis=1)


In [6]:
df = (pd.concat(dfs, ignore_index=True))

In [7]:
def uf_to_regiao(uf):
    """
        Dado o UF passado nós dizemos a qual região ele pertence
    """

    if uf in ['GO', 'MT', 'MS', 'DF']:
        return "Centro-Oeste"
    elif uf in ['MA', 'CE', 'RN', 'PE', 'BA', 'AL', 'PI', 'SE', 'PB']:
        return "Nordeste"
    elif uf in ['PA', 'RO', 'AP', 'AC', 'RR', 'AM', 'TO']:
       return "Norte"
    elif uf in ['MG', 'ES', 'RJ', 'SP']:
        return "Sudeste"
    elif uf in ['PR', 'RS', 'SC']:
        return "Sul"

In [8]:
def resolve_km(x: str):
    x = x.split(',')[0].split('.')[0]
    return x

In [9]:
def get_month(date: str) -> str:

    split_slash = date.split('/')
    split_hyphen = date.split('-')

    # for split with '-'
    if len(split_hyphen) > 1:
        return split_hyphen[1]
    
    # for split with '/'
    return split_slash[1]

In [10]:
def get_days(date: str) -> str:

    split_slash = date.split('/')
    split_hyphen = date.split('-')

    # for split with '-'
    if len(split_hyphen) > 1:
        if len(split_hyphen[0]) == 4:
            return(split_hyphen[2])
        else:
            return(split_hyphen[0])
   
    # for split with '/'
    if len(split_slash[0]) == 4:
        return(split_slash[2])
    else:
        return(split_slash[0])

In [11]:
def fixing_year(year):

    if(year == '16'):
        return '20' + year
    return year

In [12]:
def get_hour(hours: str) -> str:
    return hours.split(':')[0]

In [13]:
def get_year_slash(date: str) -> str:
    
    split_slash = date.split('/')[-1]

    if(split_slash == '16'):
        return '20' + '16'
    return split_slash

In [14]:
def get_year_hif(date: str) -> str:
    
    split_slash = date.split('-')[0]

    if(split_slash == '16'):
        return '20' + '16'
    return split_slash

In [15]:
def replace_dot(x: str):
    return x.replace(',','.')

In [16]:
df.shape[0]

1562200

In [17]:
"""

Problema: algumas colunas estão sendo preenchidas com o valor '(null)'. Como se trata de uma gama baixa de dados, vou estar removendo
para não atrapalhar futuramente em uma possível análise

"""

x = list(df.columns)

for i in x:
    df.drop(df[df[i] == '(null)'].index, inplace=True)

In [18]:
df.shape[0] # 161 linhas

1562039

In [19]:
# Eliminando valores faltantes
df.dropna(axis=0, how='any', inplace=True)

In [20]:
df.shape[0] # 96.363 linhas, apesar de ser uma quantidade alta de registros apagados, não atrapalha na amostra do todo

1465676

In [21]:
"""
Problema -> df['km'] é do tipo object e possui casos como ['100','98.3','27,2']
"""
df['km'] = df['km'].astype(str).apply(replace_dot).astype(float)

In [22]:
"""
Problema -> a coluna data_inversa, além de não estar padronizada, complica demais análises que poderiam ser mais simples
"""
df['ano'] = df['data_inversa'].apply(get_year_hif)

In [23]:
# Pegando apenas alguns exemplos para ver se foi uma parte
df[['ano']].sample(10)

,ano
414571,12/12/2009
400414,13/11/2009
118189,14/12/2007
1353998,2015
1411697,2015
1072085,2013
1374960,2015
1193951,2014
274963,18/01/2009
222270,14/09/2008


In [24]:
"""
Problema -> a coluna ano, pelo fato da data_inversa não estar padronizada, acaba sobrando registros "inconsistentes"
"""
df['ano'] = df['ano'].apply(get_year_slash)

In [25]:
# Pegando de um índice para comprovar que esse preencheu o restante do anterior
df[['ano']].iloc[679770]

ano    2011
Name: 679865, dtype: str

In [26]:
df['mes'] = df['data_inversa'].apply(get_month)

In [27]:
print(df[['mes']].min())
print(df[['mes']].max())

mes    01
dtype: str
mes    12
dtype: str


In [28]:
# Como desejo criar uma série temporal na análise, preciso criar uma coluna que me facilite isso
df['ano_mes'] = pd.to_datetime(df['ano'] + '/' + df['mes'], format='mixed')

In [29]:
# Como ficou
df['ano_mes']

2         2007-08-01
3         2007-02-01
4         2007-11-01
5         2007-12-01
6         2007-03-01
             ...    
1465832   2015-12-01
1465833   2015-11-01
1465834   2015-12-01
1465835   2015-07-01
1465836   2015-07-01
Name: ano_mes, Length: 1465676, dtype: datetime64[us]

In [30]:
df['horas'] = df['horario'].apply(get_hour)

In [31]:
print(df['horas'].min())
print(df['horas'].max())

00
23


In [32]:
df['dia'] = df['data_inversa'].apply(get_days)

In [33]:
print(df['dia'].min())
print(df['dia'].max())

01
31


In [34]:
df['br'] = df['br'].astype(str)

In [35]:
df['km_int'] = df['km'].astype(str).apply(resolve_km)

In [36]:
df['regiao'] = df['uf'].apply(uf_to_regiao)

In [37]:
df.to_csv('data-treatment.csv', index=False)